# Aprendizado de Máquina — Aula prática 03

## Seleção de Modelos e Validação Cruzada

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

As duas primeiras aulas práticas terminaram devendo a mesma coisa. Na Aula 01
escolhemos o grau do polinômio olhando o erro no conjunto de teste; na Aula 02,
o $\lambda$ do Lasso do mesmo jeito. Nas duas vezes ficou dito que aquilo era
trapaça e que a resposta viria na Aula 03. Chegamos nela.

O problema é sempre o mesmo:

> **estimar o risco de um procedimento usando os dados que temos, sem nunca
> avaliar um modelo nos pontos que o treinaram.**

E de novo vamos trabalhar na população sintética da Aula 01, porque nela — e só
nela — dá para conferir a resposta: conhecemos $r(x)$ e $\sigma^2$, logo sabemos
o risco verdadeiro de cada modelo. A validação cruzada vai ser julgada contra ele.
No fim abrimos o `superconductivity.csv` e escolhemos um hiperparâmetro no escuro,
que é como a coisa acontece de verdade.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- mostrar, numericamente, que o erro de treino é um estimador otimista do risco;
- implementar *data splitting*, LOOCV e $k$-dobras **na mão**, e reencontrar com
  eles os números que o `scikit-learn` devolve;
- verificar o atalho da alavancagem que dá o LOOCV com um único ajuste;
- reconhecer a armadilha de rodar $k$-dobras sem embaralhar;
- usar `GridSearchCV` para escolher um hiperparâmetro e `cross_val_score` por
  fora para **reportar** desempenho sem se enganar;
- medir viés e variância da própria estimativa de risco em função de $k$;
- usar o *bootstrap* para pôr uma barra de erro em uma estimativa qualquer.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos desta aula são os de `sklearn.model_selection` — é o módulo
inteiro dedicado a separar dados e estimar risco. `clone` aparece uma vez só, na
Seção 9, para repetir um procedimento de ajuste do zero.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. A população da Aula 01, de volta

Mesma função de regressão, mesmo ruído, mesmo tamanho de amostra:

$$X \sim \mathrm{Unif}[-3,3], \qquad Y = \underbrace{\sin(1{,}5X) + 0{,}3X}_{r(X)} + \varepsilon,
  \qquad \varepsilon \sim N(0, 0{,}7^2).$$

O erro irredutível é $\sigma^2 = 0{,}49$: nenhum método, por melhor que seja,
consegue erro quadrático médio abaixo disso.

In [ ]:
def r(x):
    return np.sin(1.5 * x) + 0.3 * x


SIGMA = 0.7           # erro irredutivel = 0.49
A, B = -3.0, 3.0      # suporte de X
N_TR = 50             # tamanho da amostra


def amostra(n, rng):
    x = rng.uniform(A, B, size=n)
    y = r(x) + rng.normal(0, SIGMA, size=n)
    return x, y

A classe de modelos também é a mesma: polinômios de grau $p$, ajustados por
mínimos quadrados. O `StandardScaler` no meio do caminho não muda o ajuste — é
só condicionamento numérico, para que $x^{12}$ não estoure.

In [ ]:
def modelo_poly(grau):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=grau, include_bias=False)),
        ("escala", StandardScaler()),
        ("mqo", skl.LinearRegression()),
    ])


rng = np.random.default_rng(6)
x, y = amostra(N_TR, rng)
X = x.reshape(-1, 1)

grade = np.linspace(A, B, 400)
fig, ax = subplots(figsize=(5.2, 3.2))
ax.scatter(x, y, s=18, alpha=0.75, label="a amostra (n = 50)")
ax.plot(grade, r(grade), lw=2, color="crimson", label="r(x), que so nos conhecemos")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(fontsize=8)

---
## 3. Por que o erro de treino não serve

A pergunta que abre a aula: *se eu ajustar um polinômio de grau 50 a 50 pontos,
qual o erro no treino?* Vamos responder subindo o grau devagar e olhando os dois
números — o erro no treino e o erro numa amostra nova, grande, que serve de
teste.

In [ ]:
x_te, y_te = amostra(20_000, np.random.default_rng(99))
X_te = x_te.reshape(-1, 1)

graus = np.arange(1, 21)
tabela = []
for g in graus:
    m = modelo_poly(g).fit(X, y)
    tabela.append({
        "grau": g,
        "erro no treino": np.mean((y - m.predict(X)) ** 2),
        "erro no teste": np.mean((y_te - m.predict(X_te)) ** 2),
    })

tabela = pd.DataFrame(tabela).set_index("grau")
tabela.iloc[[0, 2, 4, 9, 14, 17, 18, 19]].round(4)

O erro de treino desce **monotonicamente** e o de teste, não. Um critério que dá
nota máxima ao pior modelo da lista não serve como critério — é esse o ponto
inteiro da aula.

Repare também que o erro de treino está abaixo de $\sigma^2 = 0{,}49$ para
praticamente todos os graus, inclusive os bons. Isso é impossível para o risco de
verdade, que tem $\sigma^2$ como piso. Não é sintoma de superajuste: é o
**otimismo**, e ele é sistemático e previsível. Ajustando $p+1$ parâmetros por
mínimos quadrados a $n$ pontos, gasta-se $p+1$ graus de liberdade, e o erro de
treino esperado encolhe para

$$\mathbb{E}[\mathrm{EQM}_{\text{treino}}] \approx \sigma^2\left(1 - \frac{p+1}{n}\right),$$

desde que o modelo já seja flexível o bastante para o viés ser desprezível.
Vamos conferir isso em 300 amostras.

In [ ]:
rng_treino = np.random.default_rng(31)
soma = np.zeros(len(graus))
for _ in range(300):
    xb, yb = amostra(N_TR, rng_treino)
    Xb = xb.reshape(-1, 1)
    for j, g in enumerate(graus):
        soma[j] += np.mean((yb - modelo_poly(g).fit(Xb, yb).predict(Xb)) ** 2)

comparacao = pd.DataFrame({
    "erro de treino medio": soma / 300,
    "sigma^2 (1 - (p+1)/n)": SIGMA ** 2 * (1 - (graus + 1) / N_TR),
}, index=graus)
comparacao.index.name = "grau"
comparacao.loc[[1, 2, 3, 5, 8, 10, 15, 20]].round(4)

Do grau 5 em diante as duas colunas ficam a menos de 1,5% uma da outra. Nos graus
1 a 3 o erro de treino é bem maior que a fórmula prevê, e por um motivo que a
Aula 01 já tinha nomeado: ali o modelo é rígido demais, o viés domina, e o erro
de treino está medindo falta de flexibilidade em vez de ruído.

A lição: o erro de treino não erra por acidente, erra por construção, e o
tamanho do erro é $\sigma^2(p+1)/n$ — cresce exatamente com a complexidade do
modelo. É por isso que ele não pode arbitrar entre modelos de complexidades
diferentes.

In [ ]:
fig, ax = subplots(figsize=(5.2, 3.2))
ax.plot(tabela.index, tabela["erro no treino"], "^--", ms=4, label="erro no treino")
ax.plot(tabela.index, tabela["erro no teste"], "o-", ms=4, label="erro no teste")
ax.axhline(SIGMA ** 2, ls="--", lw=1, color="green", label="sigma^2 = 0,49")
ax.set_xlabel("grau do polinomio"); ax.set_ylabel("erro quadratico medio")
ax.set_xticks(graus[::2]); ax.set_ylim(0, 2.0)
ax.legend(fontsize=8)

A pergunta que abriu a seção pedia um polinômio de grau 50 para 50 pontos. Vamos ao
grau 49 — o maior que cabe aqui sem a coluna constante — e olhar duas coisas: o erro
de treino, e o que a curva faz **entre** os pontos observados.

In [ ]:
m49 = modelo_poly(49).fit(X, y)

print(f"EQM de treino, grau 49: {np.mean((y - m49.predict(X)) ** 2):.4f}")
print(f"EQM de treino, grau  5: {np.mean((y - modelo_poly(5).fit(X, y).predict(X)) ** 2):.4f}")
print(f"EQM de teste,  grau 49: {np.mean((y_te - m49.predict(X_te)) ** 2):.1f}")

grade_fina = np.linspace(A, B, 400)
pred49 = m49.predict(grade_fina.reshape(-1, 1))
print(f"predicao na grade: de {pred49.min():.0f} a {pred49.max():.0f}")
print(f"y observado      : de {y.min():.2f} a {y.max():.2f}")

fig, ax = subplots(figsize=(5.6, 3.4))
ax.scatter(x, y, s=18, alpha=0.75, zorder=3, label="a amostra (n = 50)")
ax.plot(grade_fina, r(grade_fina), lw=2, color="crimson", label="r(x)")
ax.plot(grade_fina, pred49, lw=1.4, color="darkorange", label="polinomio de grau 49")
ax.set_ylim(-6, 6)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("cortado em y = -6: a curva desce ate -109", fontsize=9)
ax.legend(fontsize=8)

**O erro de treino não é zero — é $0{,}3049$.** No papel deveria ser: 50 pontos e 49
covariáveis ($x, x^2, \dots, x^{49}$) dão um sistema com solução exata, e a curva
deveria passar por todas as observações. Não passa, e o motivo é numérico: $x^{49}$
com $x\in[-3,3]$ varre umas vinte e quatro ordens de grandeza, a matriz perde posto
em ponto flutuante, e o `lstsq` por trás do `LinearRegression` devolve a solução de
**norma mínima** em vez da que interpola.

O que a moral da seção precisa sobreviveu intacto: $0{,}3049$ ainda é menor que o
$0{,}3580$ do grau 5. O erro de treino continua premiando o modelo mais flexível,
mesmo quando ele é indefensável.

E ele é indefensável. A amostra vive entre $-2{,}37$ e $2{,}07$; entre dois pontos
vizinhos a curva desce a $-109$. Fora da amostra o EQM é $25{,}5$ — cinquenta vezes
o erro irredutível $\sigma^2=0{,}49$. O ajuste é excelente exatamente nos 50 lugares
onde ninguém precisa dele.

---
## 4. *Data splitting*: a solução mais simples

Separe os dados em duas partes, ajuste numa e avalie na outra. É honesto: o
conjunto de validação não participou do ajuste, então a média dos erros nele é um
estimador consistente do risco.

Só que tem dois defeitos, e os dois são mensuráveis. Vamos medi-los.

In [ ]:
x_tr, x_vl, y_tr, y_vl = skm.train_test_split(x, y, test_size=0.3, random_state=0)

m = modelo_poly(5).fit(x_tr.reshape(-1, 1), y_tr)
estimativa = np.mean((y_vl - m.predict(x_vl.reshape(-1, 1))) ** 2)
print(f"treino: {len(x_tr)} pontos   validacao: {len(x_vl)} pontos")
print(f"risco estimado (grau 5): {estimativa:.4f}")

**Defeito 1: instabilidade.** Esse número dependeu da divisão sorteada. Vamos
sortear 200 divisões diferentes e olhar a distribuição das estimativas.

In [ ]:
estimativas = []
for semente in range(200):
    xa, xb, ya, yb = skm.train_test_split(x, y, test_size=0.3, random_state=semente)
    mm = modelo_poly(5).fit(xa.reshape(-1, 1), ya)
    estimativas.append(np.mean((yb - mm.predict(xb.reshape(-1, 1))) ** 2))
estimativas = np.array(estimativas)

print(f"media  das 200 estimativas: {estimativas.mean():.4f}")
print(f"desvio das 200 estimativas: {estimativas.std(ddof=1):.4f}")
print(f"minimo: {estimativas.min():.4f}   maximo: {estimativas.max():.4f}")

fig, ax = subplots(figsize=(5.2, 2.8))
ax.hist(estimativas, bins=30, color="steelblue", alpha=0.85)
ax.set_xlabel("risco estimado por data splitting (grau 5)")
ax.set_ylabel("frequencia")

A mesma amostra, o mesmo modelo, e a estimativa passeia por uma faixa larga. Quem
reportasse o mínimo estaria mentindo por baixo; quem reportasse o máximo, por
cima. E nada nos dados diz qual das 200 divisões é a "certa" — todas são
igualmente legítimas.

**Defeito 2: desperdício.** O modelo avaliado foi treinado com 35 pontos, não com
50. Não é o modelo que você vai usar no final. Com poucos dados isso importa, e
importa na direção pessimista.

A validação cruzada resolve os dois usando toda a amostra.

---
## 5. LOOCV e o atalho que sai de graça

Deixe **uma** observação de fora por vez:

$$\widehat R_{\text{LOO}} = \frac1n \sum_{i=1}^n \big(Y_i - g_{-i}(X_i)\big)^2 .$$

Custa $n$ ajustes. Com $n=50$ isso é trivial; com $n=10^5$, não. Mas para
qualquer ajuste **linear** existe uma fórmula fechada que devolve o LOOCV exato a
partir de um **único** ajuste, usando a diagonal da matriz de projeção
$\bm H = \bm Z(\bm Z^\top \bm Z)^{-1}\bm Z^\top$:

$$\widehat R_{\text{LOO}}
  = \frac1n \sum_{i=1}^n \left(\frac{Y_i - \widehat g(X_i)}{1 - h_{ii}}\right)^2 .$$

Não vamos aceitar isso de graça — vamos conferir.

In [ ]:
GRAU = 5
Z = PolynomialFeatures(degree=GRAU).fit_transform(X)   # inclui a coluna de 1s
H = Z @ np.linalg.pinv(Z)
h = np.diag(H)
y_ajustado = H @ y

atalho = np.mean(((y - y_ajustado) / (1 - h)) ** 2)

forca_bruta = 0.0
for i in range(N_TR):
    fora = np.arange(N_TR) != i
    beta = np.linalg.lstsq(Z[fora], y[fora], rcond=None)[0]
    forca_bruta += (y[i] - Z[i] @ beta) ** 2
forca_bruta /= N_TR

print(f"LOOCV por forca bruta (50 ajustes): {forca_bruta:.10f}")
print(f"LOOCV pelo atalho     (1 ajuste)  : {atalho:.10f}")
print(f"diferenca: {abs(atalho - forca_bruta):.3e}")

Coincidem até a precisão da máquina. Vale ler o que a fórmula está dizendo: o
resíduo de validação é o resíduo de treino **inflado** por $1/(1-h_{ii})$. E
$h_{ii}$ — a *alavancagem* — mede o quanto a observação $i$ puxa a própria
predição. Quanto mais alavancada a observação, mais o resíduo de treino a
subestima.

Dá para ver isso: a alavancagem é maior nas pontas do intervalo, onde há menos
vizinhos para segurar a curva.

In [ ]:
fig, ax = subplots(figsize=(5.2, 2.9))
ax.scatter(x, h, s=20, color="darkorange")
ax.axhline((GRAU + 1) / N_TR, ls="--", lw=1, color="gray",
           label="media = (p+1)/n")
ax.set_xlabel("x"); ax.set_ylabel("alavancagem  h_ii")
ax.legend(fontsize=8)

mais_alavancados = np.argsort(h)[-3:]
print("os 3 pontos mais alavancados estao em x =", np.round(x[mais_alavancados], 2))
print(f"neles, o fator de inflacao 1/(1-h) vale "
      f"{np.round(1 / (1 - h[mais_alavancados]), 2)}")

---
## 6. $k$ dobras, na mão e no `scikit-learn`

Embaralhe, corte em $k$ lotes, e use cada lote uma vez como validação. Cada
observação é validada exatamente uma vez e treina $k-1$ vezes. Primeiro na mão,
para não haver mistério:

In [ ]:
dobras = skm.KFold(5, shuffle=True, random_state=0)

erros_por_dobra = []
for treino, validacao in dobras.split(X):
    mm = modelo_poly(GRAU).fit(X[treino], y[treino])
    erro = np.mean((y[validacao] - mm.predict(X[validacao])) ** 2)
    erros_por_dobra.append(erro)
    print(f"treino: {len(treino):2d} pts   validacao: {len(validacao):2d} pts"
          f"   EQM = {erro:.4f}")

na_mao = np.mean(erros_por_dobra)
print(f"\nCV de 5 dobras, na mao: {na_mao:.4f}")

Agora a mesma coisa com uma linha. Duas coisas para reparar no resultado: o
`scikit-learn` segue a convenção *"maior é melhor"*, então o EQM aparece
**negado** — daí o sinal de menos; e o objeto `dobras` é o mesmo, com a mesma
semente, então o corte é idêntico e os números têm de bater exatamente.

In [ ]:
pontuacoes = skm.cross_val_score(modelo_poly(GRAU), X, y, cv=dobras,
                                 scoring="neg_mean_squared_error")
print("EQM por dobra:", np.round(-pontuacoes, 4))
print(f"do sklearn: {-pontuacoes.mean():.4f}   na mao: {na_mao:.4f}")

Um detalhe que quase nunca aparece. O laço que você escreveu guardou um
$\mathrm{EQM}_j$ por dobra e tirou a média dos cinco — e é exatamente isso que o
`cross_val_score` faz, e é a fórmula 5.3 do [ISLP]. Já a Equação 4 da nota, que é a
do [AME], soma os $n$ erros e divide por $n$. Agrupando por dobra, ela vira

$$\widehat R_{k\text{-CV}} = \sum_{j=1}^{k} \frac{|L_j|}{n}\,\mathrm{EQM}_j,$$

a mesma média dos mesmos $\mathrm{EQM}_j$, só que **ponderada pelo tamanho das
dobras**. Quando $k$ divide $n$ todos os pesos valem $1/k$ e as duas coincidem
exatamente — é o caso aqui ($50 = 5 \times 10$), e é por isso que os números
bateram.

Quando $k$ não divide $n$, elas diferem. Vale medir quanto.

In [ ]:
def duas_contas(X, y, grau, cv):
    """(a) soma os n erros e divide por n;  (b) media das medias das dobras."""
    soma, mses = 0.0, []
    for tr, te in cv.split(X):
        e2 = (y[te] - modelo_poly(grau).fit(X[tr], y[tr]).predict(X[te])) ** 2
        soma += e2.sum()
        mses.append(e2.mean())
    return soma / len(y), float(np.mean(mses))


cv7 = skm.KFold(7, shuffle=True, random_state=0)      # 50 nao e multiplo de 7
agrupada, media_dobras = duas_contas(X, y, GRAU, cv7)

print("tamanhos das dobras:", [len(te) for _, te in cv7.split(X)])
print(f"(a) somando os n erros e dividindo por n: {agrupada:.6f}")
print(f"(b) media das medias das dobras         : {media_dobras:.6f}")
print(f"    o cross_val_score faz a (b)         : "
      f"{-skm.cross_val_score(modelo_poly(GRAU), X, y, cv=cv7, scoring='neg_mean_squared_error').mean():.6f}")
print(f"    diferenca relativa                  : "
      f"{abs(agrupada - media_dobras) / media_dobras:.2%}")

In [ ]:
graus_d = np.arange(1, 11)
rng_d = np.random.default_rng(11)

discordam = 0
for b in range(300):
    xb, yb = amostra(N_TR, rng_d)
    Xb = xb.reshape(-1, 1)
    cv_b = skm.KFold(7, shuffle=True, random_state=b)
    par = np.array([duas_contas(Xb, yb, g, cv_b) for g in graus_d])
    if graus_d[np.argmin(par[:, 0])] != graus_d[np.argmin(par[:, 1])]:
        discordam += 1

print(f"em 300 amostras, escolhendo o grau por uma e por outra formula,")
print(f"elas discordam em {discordam} delas")

Com $k=7$ as dobras saem com 8 e 7 observações — desequilíbrio de 14% entre a maior
e a menor — e as duas contas divergem em $0{,}9\%$. O desequilíbrio nunca passa de
**uma** observação, então encolhe como $k/n$: nos 21.263 pontos do
`superconductivity.csv` ele seria de 0,03%, e a divergência entre as fórmulas,
invisível.

Mas *quase sempre desprezível* não é *sempre*. Escolhendo o grau do polinômio por
uma e por outra, em 300 amostras, elas discordam em **9**. Não é que uma esteja
certa e a outra errada — as duas são estimativas legítimas do mesmo risco. É que
perto do mínimo a curva de CV é plana, e 1% de diferença já basta para trocar o
argmin: o mesmo fenômeno da Seção 7, visto por outro ângulo.

A regra prática que sai daqui é barata: **com $n$ pequeno, escolha um $k$ que divida
$n$** — e a pergunta não se coloca.

### A armadilha do `shuffle`

`KFold(5)` sem `shuffle=True` corta os dados **na ordem em que estão no arquivo**.
Se o banco veio ordenado por alguma variável — e quem montou o banco pode ter
ordenado por um motivo que você não conhece —, cada dobra vira um pedaço
sistemático do domínio, e o modelo é obrigado a **extrapolar** para prever nela.

Vamos simular exatamente isso: ordenar a amostra por $x$ e rodar as duas versões.

In [ ]:
ordem = np.argsort(x)
X_ord, y_ord = X[ordem], y[ordem]

sem = -skm.cross_val_score(modelo_poly(GRAU), X_ord, y_ord, cv=skm.KFold(5),
                           scoring="neg_mean_squared_error")
com = -skm.cross_val_score(modelo_poly(GRAU), X_ord, y_ord,
                           cv=skm.KFold(5, shuffle=True, random_state=0),
                           scoring="neg_mean_squared_error")

print("banco ordenado por x, KFold(5) SEM shuffle:")
print("   EQM por dobra:", np.round(sem, 2))
print(f"   media: {sem.mean():.3f}")
print("\nos MESMOS dados, KFold(5) COM shuffle:")
print("   EQM por dobra:", np.round(com, 3))
print(f"   media: {com.mean():.3f}")

Os dados são os mesmos, o modelo é o mesmo, e as duas estimativas do risco não
estão nem na mesma ordem de grandeza. Sem embaralhar, o procedimento avaliado
deixou de ser *"prever $Y$ a partir de $X$"* e virou *"extrapolar para uma região
de $x$ que o treino nunca viu"* — um problema diferente, e muito mais difícil.

Em `train_test_split` o `shuffle=True` já é o padrão; em `KFold`, **não é**.
Construa o `KFold` explicitamente e ligue o `shuffle`.

---
## 7. A validação cruzada funciona?

Agora a pergunta que só a simulação responde: a curva de CV, calculada a partir
de **uma** amostra de 50 pontos, encontra o mesmo mínimo que o risco verdadeiro?

Para saber o risco verdadeiro precisamos de muitas amostras. E há um truque que
vale a pena conhecer: não é preciso sortear ruído de teste. Como

$$\E\big[(Y - \widehat r(x))^2 \mid X = x\big] = \big(r(x) - \widehat r(x)\big)^2 + \sigma^2,$$

basta comparar a predição com $r$ e somar $\sigma^2$ no fim. Isso elimina uma
fonte inteira de variabilidade de Monte Carlo, e as curvas saem limpas com muito
menos repetições.

In [ ]:
def risco_verdadeiro(graus, n_rep=300, semente=5, n_grade=400):
    """Risco de cada grau, por Monte Carlo sobre amostras de treino."""
    rng = np.random.default_rng(semente)
    x0 = np.linspace(A, B, n_grade)
    r0 = r(x0)
    X0 = x0.reshape(-1, 1)

    somas = np.zeros(len(graus))
    for _ in range(n_rep):
        xb, yb = amostra(N_TR, rng)
        Xb = xb.reshape(-1, 1)
        for j, g in enumerate(graus):
            pred = modelo_poly(g).fit(Xb, yb).predict(X0)
            somas[j] += np.mean((pred - r0) ** 2)
    return somas / n_rep + SIGMA ** 2


graus = np.arange(1, 11)
verdade = risco_verdadeiro(graus)
print("risco verdadeiro por grau:", np.round(verdade, 3))

E agora o que o analista realmente tem: uma amostra, e mais nada.

In [ ]:
cv_media, cv_ep, treino = [], [], []
for g in graus:
    p = -skm.cross_val_score(modelo_poly(g), X, y, cv=dobras,
                             scoring="neg_mean_squared_error")
    cv_media.append(p.mean())
    cv_ep.append(p.std(ddof=1) / np.sqrt(len(p)))
    treino.append(np.mean((y - modelo_poly(g).fit(X, y).predict(X)) ** 2))

cv_media, cv_ep = np.array(cv_media), np.array(cv_ep)

g_cv = graus[np.argmin(cv_media)]
g_verdade = graus[np.argmin(verdade)]
print(f"minimo do risco verdadeiro : grau {g_verdade}")
print(f"minimo da validacao cruzada: grau {g_cv}")

In [ ]:
fig, ax = subplots(figsize=(5.6, 3.4))
ax.plot(graus, verdade, "o-", ms=4, color="crimson",
        label="risco verdadeiro (inacessivel)")
ax.errorbar(graus, cv_media, yerr=cv_ep, fmt="s-", ms=4, capsize=2.5,
            color="steelblue", label="validacao cruzada, 5 dobras")
ax.plot(graus, treino, "^--", ms=4, color="gray", label="erro de treino")
ax.axhline(SIGMA ** 2, ls="--", lw=1, color="green", label="sigma^2")
ax.axvline(g_cv, ls=":", lw=1, color="steelblue")
ax.set_xlabel("grau do polinomio"); ax.set_ylabel("erro quadratico medio")
ax.set_xticks(graus); ax.set_ylim(0.15, 1.65)
ax.legend(fontsize=7.5)

Três leituras dessa figura, em ordem de importância.

1. **A CV acerta o lugar.** Ela põe o mínimo no grau 4, o risco verdadeiro põe no
   grau 5. Um grau de diferença, com uma amostra de 50 pontos.
2. **A CV erra o nível, e erra para os dois lados.** Nos graus baixos ela fica
   abaixo da curva vinho; nos graus 8 e 9, bem acima. E isso *não é um problema*
   para seleção de modelos: o que importa é **onde** está o mínimo, não quanto
   vale. É exatamente a observação que o [ISLP] faz na Figura 5.6, em que as
   curvas de CV ora subestimam, ora superestimam o risco, e todas identificam
   corretamente o nível de flexibilidade certo.
3. **A CV avisa quando está insegura.** As barras de um erro-padrão se abrem à
   direita, exatamente onde os modelos ficam instáveis.

Se você quiser *reportar* o desempenho, aí sim precisa do valor — e para isso não
serve a mesma CV que escolheu o modelo. É o assunto da Seção 9.

### A regra de um erro-padrão

Como a estimativa tem erro-padrão, os graus 3 a 6 estão empatados dentro do
ruído. A **regra de 1-EP** resolve o empate pelo lado da parcimônia: escolha o
modelo mais simples cujo risco estimado esteja a no máximo um erro-padrão do
mínimo.

In [ ]:
limite = cv_media.min() + cv_ep[np.argmin(cv_media)]
g_1ep = graus[np.argmax(cv_media <= limite)]
print(f"minimo da CV: grau {g_cv}  (EQM {cv_media.min():.4f})")
print(f"limite de 1 erro-padrao: {limite:.4f}")
print(f"escolha pela regra de 1-EP: grau {g_1ep}")

Entre o grau 4 e o grau 5, na figura acima, o EQM de validação cruzada difere em
$0{,}003$. Vale perguntar quanto disso é sinal. Refazemos a curva inteira em outras
três amostras, mudando só a semente da Seção 2.

In [ ]:
for semente in (6, 7, 8, 9):
    rng_s = np.random.default_rng(semente)
    x_s, y_s = amostra(N_TR, rng_s)
    media_s = np.array([
        -skm.cross_val_score(modelo_poly(g), x_s.reshape(-1, 1), y_s, cv=dobras,
                             scoring="neg_mean_squared_error").mean()
        for g in graus])
    marca = "  <- a amostra deste notebook" if semente == 6 else ""
    print(f"semente {semente}: grau escolhido {graus[np.argmin(media_s)]:2d}   "
          f"EQM minimo {media_s.min():.4f}{marca}")

Quatro amostras da mesma população, três respostas diferentes: graus **4, 5, 5 e 6**.

Compare as duas grandezas. Dentro de uma amostra, a distância entre o grau 4 e o
grau 5 é de $0{,}003$ em EQM. Entre amostras, o grau escolhido anda dois degraus
inteiros. **A diferença entre modelos vizinhos não sobrevive à troca de amostra.**

É esse o argumento a favor da regra de 1-EP. Quando o empate está dentro do ruído,
desempatar por parcimônia não é mais arbitrário que desempatar pelo mínimo — e é
mais estável, porque o modelo mais simples da faixa de empate muda menos de amostra
para amostra do que o argmin.

---
## 8. Quantas dobras? O argumento clássico, e o que a medição diz

O argumento padrão para escolher $k$ é ele mesmo um balanço viés–variância, mas
**da estimativa do risco**, e está em todos os livros. O [ISLP] §5.1.4 o escreve
assim:

- com $k$ grande (LOOCV), cada modelo treina com quase $n$ observações, então a
  estimativa tem **pouco viés**; mas os $n$ modelos são treinados em conjuntos
  quase idênticos, suas predições ficam muito correlacionadas, e *"a média de
  quantidades muito correlacionadas tem variância maior que a média de
  quantidades pouco correlacionadas"* — logo, **variância alta**;
- com $k$ pequeno, o oposto: mais viés, menos variância.

Conclusão do livro: $k=5$ ou $k=10$, no meio do caminho.

O argumento é bom. Vamos medi-lo. Repetimos 300 vezes: sorteia uma amostra de 50
pontos, estima o risco do polinômio de grau 5 com cada $k$, e compara com o risco
verdadeiro daquele procedimento.

In [ ]:
alvo = risco_verdadeiro(np.array([5]), n_rep=300, semente=9)[0]
print(f"risco verdadeiro do procedimento (grau 5, n=50): {alvo:.4f}")

In [ ]:
ks = [2, 5, 10, 25, N_TR]          # N_TR dobras = LOOCV
rng = np.random.default_rng(8)

estimativas = {k: [] for k in ks}
for b in range(300):
    xb, yb = amostra(N_TR, rng)
    Xb = xb.reshape(-1, 1)
    for k in ks:
        cv = (skm.KFold(k, shuffle=True, random_state=b) if k < N_TR
              else skm.KFold(N_TR))
        s = -skm.cross_val_score(modelo_poly(5), Xb, yb, cv=cv,
                                 scoring="neg_mean_squared_error")
        estimativas[k].append(s.mean())

resumo = pd.DataFrame({
    "k": ks,
    "media": [np.mean(estimativas[k]) for k in ks],
    "vies": [np.mean(estimativas[k]) - alvo for k in ks],
    "desvio-padrao": [np.std(estimativas[k], ddof=1) for k in ks],
    "ajustes": ks,
}).set_index("k")
resumo.round(4)

> **A lição.** O viés se comporta exatamente como o livro prevê: cai de $0{,}43$
> em $k=2$ para $0{,}006$ em $k=10$, e treinar com 25 pontos em vez de 50 é
> mesmo coisa muito diferente. Já o desvio-padrão **não cresce com $k$** aqui:
> vai de $2{,}20$ em $k=2$ a $0{,}13$ na LOOCV, decrescendo o tempo todo.
>
> Não é que o argumento da correlação esteja errado — ele é um argumento sobre
> **um dos termos**. Só que o outro termo, o efeito de treinar com poucos dados,
> é maior neste problema e domina a soma. A recomendação prática sobrevive
> intacta, mas por outro motivo: de $k=5$ em diante não há praticamente nada a
> ganhar em precisão, e o custo continua subindo linearmente. Daí $k=5$ ou $10$.

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(7.4, 2.9))
pos = np.arange(len(ks))
rotulos = [str(k) if k < N_TR else f"{N_TR}\n(LOOCV)" for k in ks]

ax1.plot(pos, np.abs(resumo["vies"]), "o-", ms=4, color="steelblue", label="|vies|")
ax1.plot(pos, resumo["desvio-padrao"], "s-", ms=4, color="crimson",
         label="desvio-padrao")
ax1.set_xticks(pos); ax1.set_xticklabels(rotulos)
ax1.set_yscale("log")
ax1.set_xlabel("numero de dobras k"); ax1.set_ylabel("erro da estimativa")
ax1.set_title("precisao: estabiliza a partir de k=5", fontsize=9)
ax1.legend(fontsize=8)

ax2.plot(pos, ks, "^-", ms=4, color="green")
ax2.set_xticks(pos); ax2.set_xticklabels(rotulos)
ax2.set_xlabel("numero de dobras k"); ax2.set_ylabel("ajustes por estimativa")
ax2.set_title("custo: linear em k", fontsize=9)

O argumento do livro diz que a variância da estimativa cresce com $k$, e a medição
acima mostrou o contrário. Cabe uma objeção: com $n=50$, treinar em 25 pontos é
brutalmente diferente de treinar em 50, e esse efeito pode estar encobrindo o da
correlação. Vamos afrouxá-lo — o mesmo experimento com $n=200$.

In [ ]:
def risco_do_procedimento(n_tr, grau=5, n_rep=150, semente=9):
    """Risco verdadeiro do procedimento, pelo truque da Secao 7."""
    rng_v = np.random.default_rng(semente)
    x0 = np.linspace(A, B, 400)
    r0, X0 = r(x0), x0.reshape(-1, 1)
    soma = 0.0
    for _ in range(n_rep):
        xb, yb = amostra(n_tr, rng_v)
        soma += np.mean((modelo_poly(grau).fit(xb.reshape(-1, 1), yb).predict(X0) - r0) ** 2)
    return soma / n_rep + SIGMA ** 2


def vies_e_desvio(n_tr, ks_n, n_rep=150, semente=8):
    """Vies e desvio-padrao da CV como estimador do risco, para cada k."""
    rng_k = np.random.default_rng(semente)
    est = {k: [] for k in ks_n}
    for b in range(n_rep):
        xb, yb = amostra(n_tr, rng_k)
        Xb = xb.reshape(-1, 1)
        for k in ks_n:
            cv = (skm.KFold(k, shuffle=True, random_state=b) if k < n_tr
                  else skm.KFold(n_tr))
            est[k].append(-skm.cross_val_score(modelo_poly(5), Xb, yb, cv=cv,
                                               scoring="neg_mean_squared_error").mean())
    return est


for n_tr in (50, 200):
    ks_n = [2, 5, 10, 25, n_tr]
    alvo_n = risco_do_procedimento(n_tr)
    est = vies_e_desvio(n_tr, ks_n)
    print(f"n = {n_tr}   (risco verdadeiro {alvo_n:.4f})")
    for k in ks_n:
        rotulo = f"{k} (LOOCV)" if k == n_tr else str(k)
        print(f"   k = {rotulo:>10}   vies {np.mean(est[k]) - alvo_n:+.4f}"
              f"   desvio-padrao {np.std(est[k], ddof=1):.4f}")

**Não passa a crescer.** Com $n=200$ o desvio-padrão vai de $0{,}068$ em $k=2$ a
$0{,}057$ na LOOCV: ainda decrescendo, e agora quase plano.

O que mudou foi a escala, não o sinal. Com $n=50$ a faixa ia de $2{,}76$ a $0{,}13$
— um fator de vinte. Com $n=200$ vai de $0{,}068$ a $0{,}057$, fator $1{,}2$.
Quadruplicar a amostra não fez o termo de correlação aparecer: fez o termo de
tamanho de treino encolher, e a soma dos dois continuou descendo.

O viés conta a mesma história. Em $n=50$ ele valia $+0{,}46$ com $k=2$; em $n=200$,
$+0{,}017$. Treinar em metade dos dados deixa de doer quando a metade já é bastante.

A recomendação prática sai reforçada, e pelo motivo que a seção anterior já tinha
identificado: de $k=5$ em diante não há praticamente nada a ganhar em precisão, em
nenhum dos dois regimes, e cada dobra a mais custa um ajuste a mais.

---
## 9. Escolher um hiperparâmetro, e reportar o desempenho

`GridSearchCV` faz o laço da Seção 7 por você: para cada valor da grade, roda a
CV; guarda o melhor; e reajusta o modelo vencedor em **todos** os dados. Esse
último passo é importante e é fácil esquecer que existe: a CV estima o risco de
um *procedimento*, e o modelo que você vai usar é o treinado com a amostra
inteira.

In [ ]:
busca = skm.GridSearchCV(
    modelo_poly(1),                       # o grau vem da grade
    {"poly__degree": np.arange(1, 11)},
    cv=dobras,
    scoring="neg_mean_squared_error",
)
busca.fit(X, y)

print("grau escolhido:", busca.best_params_["poly__degree"])
print(f"EQM de CV no vencedor: {-busca.best_score_:.4f}")
print("modelo final reajustado em todos os", N_TR, "pontos:",
      type(busca.best_estimator_).__name__)

### O erro que quase todo mundo comete

Aquele `-busca.best_score_` é **o mínimo de dez estimativas ruidosas**. Um mínimo
de várias variáveis aleatórias é enviesado para baixo — é o mesmo pecado do erro
de treino, um andar acima. Usar esse número para reportar desempenho é se
enganar.

A correção é a **validação cruzada aninhada**: um laço externo estima o
desempenho, e dentro de cada dobra externa um laço interno escolhe o
hiperparâmetro do zero.

Uma comparação em cima de uma única amostra não decide nada — precisamos da média
sobre muitas. Vamos repetir 60 vezes: sortear uma amostra, calcular as duas
estimativas, e confrontar as médias com o risco verdadeiro do procedimento
inteiro *"escolher o grau por CV e reajustar"*, que a simulação entrega.

In [ ]:
externa = skm.KFold(5, shuffle=True, random_state=1)
rng = np.random.default_rng(23)

otimista, aninhado, escolhidos = [], [], []
for _ in range(60):
    xb, yb = amostra(N_TR, rng)
    Xb = xb.reshape(-1, 1)
    b = clone(busca).fit(Xb, yb)
    otimista.append(-b.best_score_)
    escolhidos.append(b.best_params_["poly__degree"])
    aninhado.append(-skm.cross_val_score(clone(busca), Xb, yb, cv=externa,
                                         scoring="neg_mean_squared_error").mean())

In [ ]:
def riscos_do_procedimento(estimador, n_rep=150, semente=17):
    """Risco de cada ajuste do procedimento, em n_rep amostras novas."""
    rng = np.random.default_rng(semente)
    x0 = np.linspace(A, B, 400)
    r0 = r(x0)
    X0 = x0.reshape(-1, 1)
    saida, graus_escolhidos = [], []
    for _ in range(n_rep):
        xb, yb = amostra(N_TR, rng)
        m = clone(estimador).fit(xb.reshape(-1, 1), yb)
        saida.append(np.mean((m.predict(X0) - r0) ** 2) + SIGMA ** 2)
        graus_escolhidos.append(m.best_params_["poly__degree"])
    return np.array(saida), np.array(graus_escolhidos)


verdadeiros, graus_verdade = riscos_do_procedimento(busca)

print(f"(a) media de -best_score_ : {np.mean(otimista):.4f}   <- otimista")
print(f"(b) media da CV aninhada  : {np.mean(aninhado):.4f}")
print(f"(c) risco verdadeiro      : {verdadeiros.mean():.4f}   <- so a simulacao entrega")
print(f"\ngraus escolhidos nas 60 amostras: "
      f"{dict(zip(*np.unique(escolhidos, return_counts=True)))}")

A CV aninhada fecha cerca de **metade** da distância entre a estimativa ingênua e
a verdade — e continua otimista. Isso é menos do que o discurso usual sugere, e
vale entender de onde vem o resto da lacuna.

In [ ]:
print(f"media  dos 150 riscos: {verdadeiros.mean():.4f}")
print(f"mediana dos 150 riscos: {np.median(verdadeiros):.4f}")
print(f"os 5 piores: {np.sort(verdadeiros)[-5:].round(2)}\n")

por_grau = pd.DataFrame({"grau": graus_verdade, "risco": verdadeiros}).groupby("grau")
print(por_grau.agg(vezes=("risco", "size"), mediana=("risco", "median"),
                   pior=("risco", "max"),
                   soma=("risco", "sum")).round(3))
print(f"\ncontribuicao dos graus >= 8 para a media: "
      f"{verdadeiros[graus_verdade >= 8].sum() / verdadeiros.sum():.1%} do total, "
      f"em {(graus_verdade >= 8).mean():.1%} das amostras")
print(f"media SEM as 2 piores amostras: {np.sort(verdadeiros)[:-2].mean():.4f}"
      f"   (a CV aninhada deu {np.mean(aninhado):.4f})")

fig, ax = subplots(figsize=(5.4, 2.9))
ax.hist(np.log10(verdadeiros), bins=30, color="crimson", alpha=0.8)
ax.axvline(np.log10(np.median(verdadeiros)), color="black", lw=1.5, label="mediana")
ax.axvline(np.log10(verdadeiros.mean()), color="steelblue", lw=1.5, label="media")
ax.set_xlabel("log10 do risco do modelo escolhido")
ax.set_ylabel("frequencia")
ax.legend(fontsize=8)

> **A lição.** A distribuição é violentamente assimétrica: a **mediana** do risco
> é cerca de $0{,}60$, mas a **média** passa de $0{,}80$, puxada por **duas** das
> 150 amostras, em que a CV escolheu um grau alto, o polinômio explodiu e o risco
> passou de 10.
>
> Tire essas duas e a média cai para $0{,}63$ — agora **abaixo** da CV aninhada.
> Isto é, no corpo da distribuição a CV aninhada é até um pouco *pessimista*, que
> é exatamente o que a teoria prevê: cada dobra externa treina o procedimento com
> 40 pontos, não com 50.
>
> Então a leitura correta não é "a CV aninhada ainda é otimista". É: **a CV
> aninhada estima muito bem o risco típico, e não enxerga a cauda.** Nenhum
> método que só olha 50 pontos enxerga — o desastre não deixa rastro na amostra
> que o produziu. Quando o risco é resumido por uma *média*, e a distribuição tem
> uma cauda dessas, a média deixa de descrever o que costuma acontecer.

Em dados reais você não tem (c), então a regra prática permanece: **se a CV
escolheu alguma coisa, ela não pode mais estimar o desempenho da coisa
escolhida.** Ou você aninha, ou separa um conjunto de teste que só é tocado uma
vez, no fim.

E a tabela por grau sugere uma segunda providência, mais barata que qualquer uma
dessas: **não ofereça à busca opções que você sabe que são ruins.** Os graus 8, 9
e 10 foram escolhidos em menos de 9% das amostras e respondem por 23% da soma dos
riscos. A grade é uma escolha sua — encurtá-la é a maneira mais barata de
encurtar a cauda.

---
## 10. O *bootstrap*: barra de erro para qualquer estimativa

A validação cruzada estima o *risco*. O bootstrap responde a outra pergunta:
**quanta incerteza há em uma estimativa qualquer?** A ideia é reamostrar dos
próprios dados, com reposição, imitando o ato de coletar amostras novas.

O exemplo do [ISLP] §5.2: você divide um investimento entre dois ativos com
retornos $X$ e $Y$, pondo a fração $\alpha$ em $X$. A fração que minimiza a
variância da carteira é

$$\alpha = \frac{\sigma_Y^2 - \sigma_{XY}}{\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY}}.$$

Na prática você estima as três variâncias dos dados e obtém $\widehat\alpha$.
Qual o erro-padrão de $\widehat\alpha$? É uma razão de estimativas
correlacionadas — boa sorte derivando isso à mão.

Vamos simular retornos com covariância conhecida, para que $\alpha$ verdadeiro
seja conhecido e o bootstrap possa ser julgado.

In [ ]:
var_x, var_y, cov_xy = 1.00, 1.25, 0.50
Sigma = np.array([[var_x, cov_xy], [cov_xy, var_y]])
alpha_verdadeiro = (var_y - cov_xy) / (var_x + var_y - 2 * cov_xy)
print(f"alpha verdadeiro: {alpha_verdadeiro:.4f}")


def alpha_de(dados):
    S = np.cov(dados, rowvar=False)
    return (S[1, 1] - S[0, 1]) / (S[0, 0] + S[1, 1] - 2 * S[0, 1])


rng = np.random.default_rng(2)
n = 100
carteira = rng.multivariate_normal([0, 0], Sigma, size=n)
alpha_chapeu = alpha_de(carteira)
print(f"alpha estimado nos {n} pontos que temos: {alpha_chapeu:.4f}")

O bootstrap, em cinco linhas: reamostre com reposição, recalcule, repita.

In [ ]:
B = 2000
replicas = np.array([alpha_de(carteira[rng.integers(0, n, n)]) for _ in range(B)])
ep_bootstrap = replicas.std(ddof=1)

# a verdade, que so temos porque a populacao e nossa:
amostras_novas = np.array([alpha_de(rng.multivariate_normal([0, 0], Sigma, size=n))
                           for _ in range(B)])
ep_verdadeiro = amostras_novas.std(ddof=1)

print(f"EP pelo bootstrap (reamostrando os 100 pontos): {ep_bootstrap:.4f}")
print(f"EP verdadeiro (sorteando 100 pontos novos)    : {ep_verdadeiro:.4f}")
print(f"IC bootstrap de 95%: "
      f"[{np.percentile(replicas, 2.5):.3f}, {np.percentile(replicas, 97.5):.3f}]"
      f"   (alpha verdadeiro = {alpha_verdadeiro:.3f})")

In [ ]:
fig, ax = subplots(figsize=(5.4, 3.0))
ax.hist(amostras_novas, bins=40, alpha=0.55, density=True,
        label="amostras novas da populacao")
ax.hist(replicas, bins=40, alpha=0.55, density=True,
        label="replicas bootstrap")
ax.axvline(alpha_verdadeiro, color="black", lw=1.5, label="alpha verdadeiro")
ax.axvline(alpha_chapeu, color="crimson", lw=1.5, ls="--", label="alpha estimado")
ax.set_xlabel("alpha"); ax.legend(fontsize=8)

As duas distribuições têm largura parecida — é isso que o bootstrap promete
entregar. Repare que a nuvem bootstrap está centrada em $\widehat\alpha$, não em
$\alpha$: o bootstrap estima a **dispersão** da estimativa, não corrige o desvio
dela. Se $\widehat\alpha$ caiu longe, o intervalo cai longe junto.

Uma última conta que vai reaparecer na Aula 06. Cada amostra bootstrap deixa de
fora, em média, uma fração $(1-1/n)^n \to e^{-1} \approx 0{,}368$ das
observações. São as *out-of-bag*, e as florestas aleatórias vão usá-las para
estimar risco de graça.

In [ ]:
sorteio = rng.integers(0, n, size=(5000, n))
fracao_fora = np.array([1 - len(np.unique(linha)) / n for linha in sorteio])
print(f"fracao media deixada de fora: {fracao_fora.mean():.4f}")
print(f"1/e = {np.exp(-1):.4f}")

---
## 11. No escuro: escolhendo o $\lambda$ da Ridge em dados reais

Fechamos com o `superconductivity.csv` da Aula 02: 21.263 materiais, 81
atributos, e a temperatura crítica como resposta. Aqui não há $r(x)$, não há
$\sigma^2$ e não há amostra nova — só a validação cruzada.

In [ ]:
import os

_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
Xs = df.drop(columns="critical_temp").values
ys = df["critical_temp"].values
print("dimensoes:", Xs.shape)

Primeiro separamos um conjunto de teste e **guardamos a chave**. Ele não vai ser
tocado até a última célula. Toda a escolha de $\lambda$ acontece dentro do
treino, por validação cruzada.

In [ ]:
X_tr, X_te, y_tr, y_te = skm.train_test_split(Xs, ys, test_size=0.3, random_state=0)

modelo = Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge())])
alphas = np.logspace(-2, 4, 25)

busca_ridge = skm.GridSearchCV(
    modelo, {"ridge__alpha": alphas},
    cv=skm.KFold(5, shuffle=True, random_state=0),
    scoring="neg_mean_squared_error",
    return_train_score=True,
)
busca_ridge.fit(X_tr, y_tr)

print(f"alpha escolhido: {busca_ridge.best_params_['ridge__alpha']:.4g}")
print(f"EQM de CV no vencedor: {-busca_ridge.best_score_:.3f}")

A curva de CV com as barras de um erro-padrão, e as duas escolhas possíveis: o
mínimo e a regra de 1-EP.

In [ ]:
res = busca_ridge.cv_results_
media = -res["mean_test_score"]
ep = res["std_test_score"] / np.sqrt(5)

i_min = np.argmin(media)
limite = media[i_min] + ep[i_min]
i_1ep = np.max(np.where(media <= limite))      # o alpha MAIOR = modelo mais simples

fig, ax = subplots(figsize=(5.4, 3.2))
ax.errorbar(alphas, media, yerr=ep, fmt="o-", ms=3.5, capsize=2.5)
ax.axvline(alphas[i_min], ls=":", color="steelblue", label="minimo da CV")
ax.axvline(alphas[i_1ep], ls="--", color="darkorange", label="regra de 1-EP")
ax.set_xscale("log")
ax.set_xlabel("alpha (= lambda)"); ax.set_ylabel("EQM de validacao cruzada")
ax.legend(fontsize=8)

print(f"minimo da CV : alpha = {alphas[i_min]:.4g}")
print(f"regra de 1-EP: alpha = {alphas[i_1ep]:.4g}")

A curva é quase plana numa faixa larga de $\lambda$ — o que quer dizer que, neste
problema, a regularização não é o que decide o desempenho. Vale registrar isso:
nem toda busca de hiperparâmetro compensa o esforço, e a curva de CV é quem
avisa.

Agora sim, a única vez em que tocamos o teste.

In [ ]:
from sklearn.metrics import mean_squared_error

eqm_teste = mean_squared_error(y_te, busca_ridge.predict(X_te))
print(f"EQM de CV no treino (otimista): {-busca_ridge.best_score_:.3f}")
print(f"EQM no teste, medido uma vez  : {eqm_teste:.3f}")
print(f"variancia de y (preditor constante): {ys.var():.3f}")

Esse número é uma medição única, e como toda medição tem incerteza. O bootstrap da
Seção 10 diz quanta. Aqui o modelo fica **fixo** — nada é reajustado —, e o que
reamostramos é o conjunto de teste em que ele é avaliado.

In [ ]:
erro2 = (y_te - busca_ridge.predict(X_te)) ** 2      # erro quadratico observacao a observacao

rng_b = np.random.default_rng(3)
replicas = np.array([erro2[rng_b.integers(0, len(erro2), len(erro2))].mean()
                     for _ in range(2000)])

lo, hi = np.percentile(replicas, [2.5, 97.5])
print(f"EQM no teste, medido uma vez: {eqm_teste:.3f}   (n = {len(y_te)})")
print(f"erro-padrao pelo bootstrap  : {replicas.std(ddof=1):.3f}")
print(f"intervalo de 95%            : [{lo:.3f}, {hi:.3f}]")
print(f"EQM de CV no treino         : {-busca_ridge.best_score_:.3f}"
      f"   {'(dentro do intervalo)' if lo <= -busca_ridge.best_score_ <= hi else '(fora)'}")

O intervalo de 95% vai de $295{,}2$ a $319{,}3$ — largura $24$, ou $\pm4\%$ do valor
medido. É estreito, e por um motivo pouco glamouroso: o teste tem $6\,379$
observações, e o erro-padrão de uma média cai com $\sqrt{n}$.

Repare onde cai o EQM de validação cruzada do treino, $313{,}0$: **dentro** do
intervalo. Não prova nada em geral, mas neste problema a CV não ficou otimista de
forma detectável — o que é coerente com a curva quase plana da célula anterior, já
que a única coisa escolhida por ela foi um $\lambda$ que quase não muda o ajuste.

E vale nomear o que este bootstrap **não** cobre. Ele mede a variabilidade de
avaliar *este* modelo em amostras de teste do mesmo tamanho. Não mede o que mudaria
se a divisão treino/teste tivesse caído em outro lugar — para isso seria preciso
reamostrar antes da divisão, e reajustar tudo em cada réplica.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| erro de treino | §3 | desce sempre; o otimismo é sistemático e vale $\sigma^2(p+1)/n$ |
| *data splitting* | §4 | honesto, mas instável (200 divisões, faixa larga) e desperdiça dados |
| LOOCV | §5 | atalho da alavancagem confere com a força bruta até $10^{-15}$ |
| $k$ dobras | §6 | na mão e no `cross_val_score` dão o mesmo número |
| `shuffle` | §6 | sem embaralhar, num banco ordenado, a estimativa muda de ordem de grandeza |
| CV × risco | §7 | erra o nível, acerta o lugar do mínimo — que é o que a seleção precisa |
| escolha de $k$ | §8 | viés cai como o livro prevê; o desvio-padrão **também** cai — medido, não previsto |
| CV aninhada | §9 | aninhar fecha metade da lacuna; o resto mora na cauda das escolhas ruins |
| bootstrap | §10 | EP da réplica reproduz o EP verdadeiro; 36,8% ficam de fora (OOB) |
| caso real | §11 | curva de CV plana: às vezes o hiperparâmetro não é o que decide |

**Leitura recomendada.** [AME] §1.4 e §1.5.1 — vale ler a Observação 1.5, que é a
conta de por que o LOOCV é aproximadamente não-viesado, e o trecho sobre
intervalos de confiança para o risco, que é o que a Seção 10 faz por bootstrap.
[ISLP] Capítulo 5: §5.1.1–5.1.3 (as três estratégias), §5.1.4 (o argumento
viés–variância que medimos na Seção 8) e §5.2 (bootstrap, com este mesmo exemplo
do investimento). As Figuras 5.6 e 5.8 do livro são a versão deles da nossa
figura da Seção 7.

**Para praticar.** `Lista teorica 03.pdf` (teórica, com gabarito) e
`Lista prática 03.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 04 troca o polinômio global por métodos que olham só a
vizinhança do ponto — KNN e regressão local. A ferramenta para escolher o $k$
deles é a que acabamos de montar aqui.